In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{ReLU} $$

$$ \mathbb{R} \to \mathbb{R}, \quad y(x) = \max(0, x) $$

$$ \mathbb{R^n} \to \mathbb{R^n}, \quad y(\mathbf{x}) = (y(x_1), y(x_2), \dots, y(x_n)) $$

$$ 
\frac{\partial y}{\partial x_i} = 
\begin{dcases}
    1 & x > 0 \\ 
    0 & x = 0 \\
    0 & x < 0
\end{dcases}
$$

$$ d\mathbf{y} = J_y(\mathbf{x}) \cdot d\mathbf{x} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dp}, d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle =
\left \langle \frac{dF}{dy}, J_y(\mathbf{p}) \, d\mathbf{p} \right \rangle =
\left \langle J_y(\mathbf{p}) ^\top \, \frac{dF}{dy}, d\mathbf{p} \right \rangle \implies
$$

$$ 
\frac{dF}{dp} = 
J_y(\mathbf{p})^\top \, \frac{dF}{dy} = 
\frac{dy}{dp} \odot \frac{dF}{dy} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def relu(x: tr.Tensor) -> tr.Tensor:
    return tr.maximum(tr.zeros_like(x), x)


class ReLUFunction(autograd.Function):
    """Custom autograd function for the `ReLU`."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return relu(x)

    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor, ]:
        (x,) = ctx.saved_tensors

        dy_dx = tr.ones_like(dF_dy)
        dy_dx[x < 0] = 0
        dy_dx[x == 0] = 0

        dF_dx = dy_dx * dF_dy
        
        return (dF_dx, )
    
    
class ReLU(nn.Module):
    """Custom module for the `ReLU` function."""
    
    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return ReLUFunction.apply(x)

In [ ]:
def test_gradcheck(samples):
    def fn(x):
        return ReLUFunction.apply(x)

    z = tr.randn(samples, 1, dtype=tr.float64, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0, dtype=tr.float64),
        tr.tensor(+1.0, dtype=tr.float64)
    )

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, 1, requires_grad=True)

    z1 = z.clone().detach().requires_grad_(True)
    y1 = ReLU()(z1).sum()
    y1.backward()

    z2 = z.clone().detach().requires_grad_(True)
    y2 = nn.ReLU()(z2).sum()
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert z1.grad == approx(z2.grad)


def test_gradcheck(samples = 10) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return ReLUFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)

if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)